In [2]:
%load_ext autoreload
%autoreload 2

import cv2
import numpy as np
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
from pathlib import Path

# Загружаем предобработанный рисунок
img_path = "OUTPUT/img01_clean.png"
rgba = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
print(f"Загружено: {rgba.shape[1]}x{rgba.shape[0]}, каналов: {rgba.shape[2]}")

plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(rgba, cv2.COLOR_BGRA2RGBA))
plt.axis("off")
plt.title("Персонаж для разметки")
plt.show()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Загружено: 396x544, каналов: 4


In [3]:
# Шаблон скелета — двуногий (подходит для Пикачу)
SKELETON = {
    'points': [
        'head',         # 0 — макушка
        'neck',         # 1 — шея
        'left_hand',    # 2 — левая лапа
        'right_hand',   # 3 — правая лапа
        'hip',          # 4 — таз
        'left_foot',    # 5 — левая стопа
        'right_foot',   # 6 — правая стопа
        'tail',         # 7 — хвост (основание)
    ],
    'bones': [
        (0, 1),  # head → neck
        (1, 2),  # neck → left_hand
        (1, 3),  # neck → right_hand
        (1, 4),  # neck → hip
        (4, 5),  # hip → left_foot
        (4, 6),  # hip → right_foot
        (4, 7),  # hip → tail
    ]
}

# --- Интерактивная разметка ---
canvas = rgba.copy()
if canvas.shape[2] == 4:
    # RGBA → BGR с белым фоном для отображения
    alpha = canvas[:, :, 3:4] / 255.0
    bgr = canvas[:, :, :3]
    white = np.full_like(bgr, 255)
    display = (bgr * alpha + white * (1 - alpha)).astype(np.uint8)
else:
    display = canvas.copy()

base = display.copy()
clicked = []

def redraw():
    global display
    display = base.copy()
    for a, b in SKELETON['bones']:
        if a < len(clicked) and b < len(clicked):
            cv2.line(display, clicked[a], clicked[b], (0, 0, 255), 2)
    for i, pt in enumerate(clicked):
        cv2.circle(display, pt, 5, (0, 0, 255), -1)
        cv2.putText(display, SKELETON['points'][i], (pt[0]+8, pt[1]-5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 200, 0), 1)
    n = len(clicked)
    total = len(SKELETON['points'])
    if n < total:
        txt = f"Click: {SKELETON['points'][n]} ({n}/{total}) | RMB=undo"
    else:
        txt = f"Done ({total}/{total})! Press any key"
    cv2.putText(display, txt, (10, 22), cv2.FONT_HERSHEY_SIMPLEX, 0.55, (0, 180, 255), 2)

def on_mouse(event, x, y, flags, param):
    if event == cv2.EVENT_LBUTTONDOWN and len(clicked) < len(SKELETON['points']):
        clicked.append((x, y))
        redraw()
        cv2.imshow('Pose Markup', display)
        if len(clicked) == len(SKELETON['points']):
            print("Готово! Нажми любую клавишу.")
    elif event == cv2.EVENT_RBUTTONDOWN and clicked:
        clicked.pop()
        redraw()
        cv2.imshow('Pose Markup', display)

redraw()
cv2.imshow('Pose Markup', display)
cv2.setMouseCallback('Pose Markup', on_mouse)
cv2.waitKey(0)
cv2.destroyAllWindows()

# Результат
keypoints = {name: pt for name, pt in zip(SKELETON['points'], clicked)}
print("\nРазмеченные точки:")
for name, (x, y) in keypoints.items():
    print(f"  {name:16s}  ({x}, {y})")


Готово! Нажми любую клавишу.

Размеченные точки:
  head              (212, 75)
  neck              (189, 245)
  left_hand         (35, 144)
  right_hand        (361, 174)
  hip               (192, 384)
  left_foot         (141, 491)
  right_foot        (267, 503)
  tail              (196, 382)


In [4]:
# Визуализация скелета поверх персонажа
vis = rgba.copy()
if vis.shape[2] == 4:
    alpha_ch = vis[:, :, 3:4] / 255.0
    bgr = vis[:, :, :3]
    white = np.full_like(bgr, 255)
    vis_rgb = (bgr * alpha_ch + white * (1 - alpha_ch)).astype(np.uint8)
    vis_rgb = cv2.cvtColor(vis_rgb, cv2.COLOR_BGR2RGB)
else:
    vis_rgb = cv2.cvtColor(vis, cv2.COLOR_BGR2RGB)

# Рисуем кости
for a, b in SKELETON['bones']:
    p1 = keypoints[SKELETON['points'][a]]
    p2 = keypoints[SKELETON['points'][b]]
    cv2.line(vis_rgb, p1, p2, (255, 50, 50), 3)

# Рисуем точки + подписи
for name, (x, y) in keypoints.items():
    cv2.circle(vis_rgb, (x, y), 7, (255, 0, 0), -1)
    cv2.circle(vis_rgb, (x, y), 7, (255, 255, 255), 2)
    cv2.putText(vis_rgb, name, (x + 10, y - 5),
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 180, 0), 2)

plt.figure(figsize=(8, 8))
plt.imshow(vis_rgb)
plt.axis("off")
plt.title("Skeleton overlay")
plt.show()

In [5]:
import json

# Сохраняем разметку
pose_data = {
    "image": img_path,
    "skeleton": SKELETON,
    "keypoints": {name: list(pt) for name, pt in keypoints.items()},
}

out_path = f"OUTPUT/{Path(img_path).stem}_pose.json"
with open(out_path, "w", encoding="utf-8") as f:
    json.dump(pose_data, f, indent=2, ensure_ascii=False)

print(f"Сохранено: {out_path}")
print(json.dumps(pose_data["keypoints"], indent=2))


Сохранено: OUTPUT/img01_clean_pose.json
{
  "head": [
    212,
    75
  ],
  "neck": [
    189,
    245
  ],
  "left_hand": [
    35,
    144
  ],
  "right_hand": [
    361,
    174
  ],
  "hip": [
    192,
    384
  ],
  "left_foot": [
    141,
    491
  ],
  "right_foot": [
    267,
    503
  ],
  "tail": [
    196,
    382
  ]
}


In [6]:
# Сохраняем визуализацию скелета
vis = rgba.copy()
alpha_ch = vis[:, :, 3:4] / 255.0
bgr = vis[:, :, :3]
white = np.full_like(bgr, 255)
vis_bgr = (bgr * alpha_ch + white * (1 - alpha_ch)).astype(np.uint8)

for a, b in SKELETON['bones']:
    p1 = keypoints[SKELETON['points'][a]]
    p2 = keypoints[SKELETON['points'][b]]
    cv2.line(vis_bgr, p1, p2, (0, 50, 255), 4)

for name, (x, y) in keypoints.items():
    cv2.circle(vis_bgr, (x, y), 8, (0, 0, 255), -1)
    cv2.circle(vis_bgr, (x, y), 8, (255, 255, 255), 2)

cv2.imwrite("OUTPUT/img01_pose.png", vis_bgr)
print(f"Saved: OUTPUT/img01_pose.png ({vis_bgr.shape[1]}x{vis_bgr.shape[0]})")


Saved: OUTPUT/img01_pose.png (396x544)
